In [ ]:
# ==================== STAGE 1: COLAB SETUP & MS-COCO DOWNLOAD ====================
import os
from google.colab import drive

# 1. Mount Google Drive (we will use this at the very end to save the model and tokenizer)
drive.mount('/content/drive')

# 2. Create local folder for MS-COCO dataset
COCO_DIR = "/content/coco"
os.makedirs(COCO_DIR, exist_ok=True)

# 3. Install fast downloader (aria2) and unzip tool
!apt-get update -qq && apt-get install -y -qq aria2 unzip

# 4. Download and extract Captions Annotations (241 MB)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q {COCO_DIR}/annotations_trainval2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/annotations_trainval2017.zip

# 5. Download and extract Train2017 Images (18 GB)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/zips/train2017.zip
!unzip -q {COCO_DIR}/train2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/train2017.zip

# 6. Download and extract Val2017 Images (1 GB - required for validation dataset)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/zips/val2017.zip
!unzip -q {COCO_DIR}/val2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/val2017.zip

print("[SUCCESS] MS-COCO Train, Val, and Annotations are ready!")

In [ ]:
# ==================== STAGE 2A: TOKENIZER FUNCTIONS ====================
import os
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LENGTH = 40

def impage_Caption_relation(img_dir, captions_json_path):
    """
    Reads the MS-COCO JSON file and matches each image path with its caption.
    Adds <start> and <end> tags to every caption.
    """
    with open(captions_json_path, "r", encoding="utf-8") as f:
        coco_Dataset = json.load(f)

    # Map image ID to its full file path
    imageid_To_path = {
        img["id"]: os.path.join(img_dir, img["file_name"])
        for img in coco_Dataset["images"]
    }

    img_paths = []
    captions  = []

    # Pair each caption annotation with its corresponding image path
    for ann in coco_Dataset["annotations"]:
        img_id = ann["image_id"]
        if img_id in imageid_To_path:
            img_paths.append(imageid_To_path[img_id])
            caption_text = f"<start> {ann['caption'].strip()} <end>"
            captions.append(caption_text)

    return img_paths, captions


def build_tokenizer(train_captions, max_words=MAX_VOCAB_SIZE):
    """
    Fits a brand-new Keras Tokenizer in memory on the training captions.
    Keeps '<' and '>' so <start> and <end> are not removed.
    """
    custom_filters = '!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n'

    tokenizer = Tokenizer(
        num_words=max_words,
        oov_token='<unk>',
        filters=custom_filters,
        lower=True
    )
    tokenizer.fit_on_texts(train_captions)
    return tokenizer


def tokenize_and_pad(captions, tokenizer, max_len=MAX_SEQ_LENGTH):
    """
    Converts text captions into integer lists and pads them with zeros at the end.
    """
    sequences = tokenizer.texts_to_sequences(captions)
    padded_seqs = pad_sequences(
        sequences,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )
    return padded_seqs


def decode_tokenizer_sequence(token_ids, tokenizer):
    """
    Converts a list of integer token IDs back into a readable English sentence.
    """
    words = []
    for idx in token_ids:
        if idx == 0:  # Skip padding zeros
            continue
        word = tokenizer.index_word.get(idx, '<unk>')
        if word == '<start>':
            continue
        if word == '<end>':
            break
        words.append(word)
    return " ".join(words)

In [ ]:
# ==================== STAGE 2B: EXECUTE TOKENIZER ====================
TRAIN_IMG_DIR = f"{COCO_DIR}/train2017"
TRAIN_JSON_PATH = f"{COCO_DIR}/annotations/captions_train2017.json"
VAL_IMG_DIR = f"{COCO_DIR}/val2017"
VAL_JSON_PATH = f"{COCO_DIR}/annotations/captions_val2017.json"

# 1. Load image paths and captions for Train and Val
train_img_paths, train_captions = impage_Caption_relation(TRAIN_IMG_DIR, TRAIN_JSON_PATH)
val_img_paths, val_captions = impage_Caption_relation(VAL_IMG_DIR, VAL_JSON_PATH)

# 2. Fit Tokenizer on training captions in memory
tokenizer = build_tokenizer(train_captions, max_words=MAX_VOCAB_SIZE)

# 3. Convert captions to padded integer sequences
padded_train_seqs = tokenize_and_pad(train_captions, tokenizer)
padded_val_seqs = tokenize_and_pad(val_captions, tokenizer)

print(f"Total Train Pairs: {len(train_img_paths)} | Total Val Pairs: {len(val_img_paths)}")

In [ ]:
# ==================== STAGE 3A-v2: PREPROCESSING FOR THE NEW PRETRAINED CNN ====================
# Why a new preprocess function?
# ResNet50 needed keras.applications.resnet50.preprocess_input (it expects BGR + mean-subtracted pixels).
# EfficientNetV2 is different: it has its OWN rescaling layer built INSIDE the model (include_preprocessing=True
# by default), so we must feed it RAW 0-255 pixels. If we still ran resnet's preprocess_input, the CNN would
# see double-normalized pixels and the "powerful pretrained features" would actually come out WORSE.

IMG_SIZE   = 224
BATCH_SIZE = 512


def preprocess_image_v2(img_path):
    """Reads an image from disk and resizes it to 224x224. No manual normalization - EfficientNetV2 does that itself."""
    raw_bytes = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(raw_bytes, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img


def prepare_caption_sample_v2(img_path, padded_seq):
    """
    Exact same Teacher Forcing shift as before:
    decoder_input  = [<start>, a, dog, <end>, 0]
    decoder_target = [a, dog, <end>, 0, 0]
    Only the image preprocessing changed (preprocess_image_v2 instead of preprocess_image_keras).
    """
    img_tensor = preprocess_image_v2(img_path)
    seq = tf.cast(padded_seq, tf.int32)

    decoder_input = seq[:-1]
    decoder_target = seq[1:]

    return (img_tensor, decoder_input), decoder_target


def build_datasets_v2(train_paths, train_seqs, val_paths, val_seqs):
    """Same tf.data pipeline as build_datasets(), just wired to the v2 preprocessing function."""
    # 1. Build Training Dataset
    train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_seqs))
    train_ds = train_ds.shuffle(buffer_size=2000, seed=42)
    train_ds = train_ds.map(prepare_caption_sample_v2, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=False)

    # 2. Build Validation Dataset
    val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_seqs))
    val_ds = val_ds.map(prepare_caption_sample_v2, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE, drop_remainder=False)

    # 3. Prefetch batches in the background for faster GPU training
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds


In [ ]:
# ==================== STAGE 3B-v2: EXECUTE DATA PIPELINE (FOR TRANSFORMER MODEL) ====================
train_ds_v2, val_ds_v2 = build_datasets_v2(
    train_img_paths,
    padded_train_seqs,
    val_img_paths,
    padded_val_seqs
)

print(f"Train batches: {len(train_ds_v2)} | Val batches: {len(val_ds_v2)}")


In [ ]:
# ==================== STAGE 4-v2: BUILD MODEL ARCHITECTURE (PRETRAINED CNN + TRANSFORMER) ====================
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetV2B0   # stronger feature extractor than ResNet50

MAX_WORDS = 10000
D_MODEL = 256            # width of every vector flowing through the Transformer (plays the same role as LSTM's 256 units)
NUM_HEADS = 8            # how many attention "views" the model uses at once
FF_DIM = 512             # hidden size inside each decoder block's feed-forward network
NUM_DECODER_LAYERS = 2   # how many Transformer decoder blocks we stack on top of each other

# ---------- 4.1 : IMAGE ENCODER (Pretrained CNN, kept as a GRID of patches instead of 1 pooled vector) ----------
inputA = Input(shape=(224, 224, 3), name="image_input")

# Load pretrained EfficientNetV2B0 and freeze its weights (same freezing idea as before, just a stronger backbone)
# pooling=None is the KEY change: old code used pooling="avg" which squashes the whole image into ONE blurry vector.
# Here we keep the full 7x7 grid of feature "patches", so the decoder can later ATTEND to different regions
# of the image for different words (e.g. look at the "dog" region when writing the word "dog").
base_cnn = EfficientNetV2B0(weights="imagenet", include_top=False, pooling=None)
base_cnn.trainable = False

# x1 shape: (batch, 7, 7, 1280) -> 7x7 = 49 patches, each describing one region of the image
x1 = base_cnn(inputA)

# Flatten the 7x7 grid into a SEQUENCE of 49 patch-vectors -> (batch, 49, 1280)
x2 = Reshape((-1, x1.shape[-1]), name="flatten_patches")(x1)

# Project every patch vector down to D_MODEL size so it matches the Transformer's internal width
encoder_output = Dense(D_MODEL, activation="relu", name="encoder_projection")(x2)   # (batch, 49, D_MODEL)
encoder_output = Dropout(0.3)(encoder_output)


# ---------- 4.2 : POSITIONAL ENCODING (gives the Transformer a sense of word order) ----------
# An LSTM reads words one-by-one so it automatically "knows" the order. A Transformer looks at every
# word at once, so we manually inject a fixed sin/cos pattern that tells it "this is word 0, this is word 1..."
def get_positional_encoding(max_len, d_model):
    positions = np.arange(max_len)[:, np.newaxis]              # (max_len, 1)
    dims = np.arange(d_model)[np.newaxis, :]                   # (1, d_model)
    angle_rates = 1 / np.power(10000, (2 * (dims // 2)) / np.float32(d_model))
    angle_rads = positions * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])   # even indices -> sin
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])   # odd indices  -> cos

    return tf.cast(angle_rads[np.newaxis, ...], dtype=tf.float32)   # (1, max_len, d_model)


# ---------- 4.3 : TEXT INPUT -> EMBEDDING + POSITIONAL ENCODING ----------
inputB = Input(shape=(None,), name="text_input")

token_embed = Embedding(input_dim=MAX_WORDS, output_dim=D_MODEL, mask_zero=True, name="token_embedding")(inputB)

# Standard Transformer trick: scale the embedding up so it isn't dwarfed by the positional encoding values
token_embed = token_embed * tf.math.sqrt(tf.cast(D_MODEL, tf.float32))

pos_encoding = get_positional_encoding(MAX_SEQ_LENGTH, D_MODEL)
seq_len = tf.shape(inputB)[1]
decoder_input = token_embed + pos_encoding[:, :seq_len, :]
decoder_input = Dropout(0.3)(decoder_input)


# ---------- 4.4 : ONE TRANSFORMER DECODER BLOCK (we stack this NUM_DECODER_LAYERS times) ----------
def transformer_decoder_block(x, encoder_output, d_model, num_heads, ff_dim, block_name):
    """
    A single Transformer decoder block does 3 things, in order:
      1. Masked Self-Attention -> each word looks only at EARLIER words (use_causal_mask=True = reading left to right)
      2. Cross-Attention       -> each word looks at the IMAGE patches. This REPLACES the old LSTM
                                   "initial_state=[x_h, x_c]" trick - instead of one fixed starting memory,
                                   every single word gets to re-look at the image.
      3. Feed-Forward Network  -> processes each position independently
    Every sub-layer has a residual connection ("x + ...") and LayerNormalization, which is what makes
    deep Transformers stable to train.
    """
    # 1. Masked Self-Attention
    self_attn = MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, name=f"{block_name}_self_attn"
    )(query=x, value=x, key=x, use_causal_mask=True)
    x = LayerNormalization(epsilon=1e-6, name=f"{block_name}_norm1")(x + self_attn)

    # 2. Cross-Attention: query = caption written so far, key/value = the 49 image patches
    cross_attn = MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads, name=f"{block_name}_cross_attn"
    )(query=x, value=encoder_output, key=encoder_output)
    x = LayerNormalization(epsilon=1e-6, name=f"{block_name}_norm2")(x + cross_attn)

    # 3. Feed-Forward Network
    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(d_model)(ffn)
    x = LayerNormalization(epsilon=1e-6, name=f"{block_name}_norm3")(x + ffn)

    return x


# ---------- 4.5 : STACK THE DECODER BLOCKS ----------
x = decoder_input
for i in range(NUM_DECODER_LAYERS):
    x = transformer_decoder_block(x, encoder_output, D_MODEL, NUM_HEADS, FF_DIM, block_name=f"decoder_block_{i+1}")

# ---------- 4.6 : FINAL PREDICTION LAYER (same idea as before: score every word in the vocab) ----------
z1 = Dropout(0.3)(x)
z2 = Dense(MAX_WORDS, activation="linear", name="predictions")(z1)

# ---------- 4.7 : BUILD THE FINAL MODEL ----------
Transformer_Model = Model(inputs=[inputA, inputB], outputs=z2)
Transformer_Model.summary()


In [ ]:
# ==================== STAGE 5-v2: COMPILE & TRAIN THE TRANSFORMER MODEL ====================
# Loss, optimizer and the masked_accuracy metric are IDENTICAL to Stage 5A - we are reusing the
# exact same masked_accuracy() and train_model() functions you already wrote, nothing new to learn here.
Transformer_Model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[masked_accuracy]
)

History_2 = train_model(Transformer_Model, train_ds_v2, val_ds_v2, epochs=15)


In [ ]:
# ==================== STAGE 6-v2: TEST THE TRANSFORMER MODEL ====================
# No changes needed to generate_1_caption() at all!
# It already rebuilds the FULL sequence and feeds it back into the model on every step (not just the
# last word) - which is exactly how a Transformer decoder expects to be called. The LSTM-era function
# works for the Transformer model too, as-is.
test_idx = 108
test_img_path = train_img_paths[test_idx]
preprocessed_img_v2 = preprocess_image_v2(test_img_path)

predicted_caption = generate_1_caption(Transformer_Model, preprocessed_img_v2, tokenizer)

plt.figure(figsize=(6, 6))
plt.imshow(Image.open(test_img_path))
plt.axis("off")
plt.show()

print(f"Generated Caption: {predicted_caption}")


In [ ]:
# ==================== STAGE 7-v2: SAVE TRANSFORMER MODEL (.keras) ====================
# Tokenizer is unchanged, so we don't need to save it again - the one from Stage 7 still works fine.
TRANSFORMER_MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "Transformer_Model.keras")
Transformer_Model.save(TRANSFORMER_MODEL_SAVE_PATH)

print(f"Transformer model saved to : {TRANSFORMER_MODEL_SAVE_PATH}")
